In [ ]:
# 1. Define the raw download URL for the v0.01 training data
# This is extracted directly from the original script's _DL_URL logic
DOWNLOAD_URL = "https://s3.amazonaws.com/datasets.huggingface.co/SpeechCommands/v0.01/v0.01_train.tar.gz"

print(f"Starting direct download of {DOWNLOAD_URL}...")

# 2. Download the file using wget
!wget -q $DOWNLOAD_URL -O speech_commands_train.tar.gz

print("Download complete. Starting extraction...")

# 3. Extract the contents of the archive into a folder named 'v0.01_train'
!mkdir v0.01_train
!tar -xzf speech_commands_train.tar.gz -C v0.01_train

print("Extraction complete. The audio files are now in the 'v0.01_train' folder.")

Starting direct download of https://s3.amazonaws.com/datasets.huggingface.co/SpeechCommands/v0.01/v0.01_train.tar.gz...
Download complete. Starting extraction...
Extraction complete. The audio files are now in the 'v0.01_train' folder.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Google Drive mounted successfully.")

Mounted at /content/drive
Google Drive mounted successfully.


In [ ]:
import os
import torch
import torch.nn as nn
import torchaudio
import torchaudio.transforms as T
from torch.utils.data import Dataset, DataLoader, random_split
from torch.optim import Adam

class AudioKwsDataset(Dataset):
    def __init__(self, root_dir, sample_rate=16000, n_mels=40, exclude_label="_unknown_"):
        self.root_dir = root_dir
        self.sample_rate = sample_rate

        mel_transform = T.MelSpectrogram(
            sample_rate=sample_rate, n_fft=400, win_length=400,
            hop_length=160, n_mels=n_mels
        )

        # Build label map
        all_dirs = sorted([
            d for d in os.listdir(root_dir)
            if os.path.isdir(os.path.join(root_dir, d))
        ])
        if exclude_label in all_dirs:
            all_dirs.remove(exclude_label)

        self.label_to_idx = {name: i for i, name in enumerate(all_dirs)}

        # ── NEW: preprocess everything into RAM ──────────────────────────
        self.cached_specs = []
        self.cached_labels = []
        self.samples = []

        total = sum(
            len([f for f in os.listdir(os.path.join(root_dir, d)) if f.endswith('.wav')])
            for d in all_dirs
        )
        print(f"Loading {total} audio files into RAM...")

        count = 0
        for label_name in all_dirs:
            dir_path = os.path.join(root_dir, label_name)
            idx = self.label_to_idx[label_name]
            for f in os.listdir(dir_path):
                if not f.endswith('.wav'):
                    continue

                path = os.path.join(dir_path, f)
                waveform, sr = torchaudio.load(path)

                # Resample + pad to exactly 1 second
                if sr != sample_rate:
                    waveform = torchaudio.functional.resample(waveform, sr, sample_rate)
                if waveform.shape[1] > sample_rate:
                    waveform = waveform[:, :sample_rate]
                else:
                    waveform = torch.nn.functional.pad(
                        waveform, (0, sample_rate - waveform.shape[1])
                    )

                spec = mel_transform(waveform)
                spec = torch.log(spec + 1e-9)


                self.cached_specs.append(spec)
                self.cached_labels.append(idx)
                self.samples.append((path, idx))

                count += 1
                if count % 5000 == 0:
                    print(f"  {count}/{total} loaded...")

        print(f"Done! All {total} spectrograms cached in RAM.")
        # ────────────────────────────────────────────────────────────────

    def __len__(self):
        return len(self.cached_labels)

    def __getitem__(self, idx):
        # Just a list lookup — no disk I/O, no FFT math
        return self.cached_specs[idx], self.cached_labels[idx]

full_dataset = AudioKwsDataset(root_dir='/content/v0.01_train')

# Cache it so re-instantiation is instant
_cached_dataset = full_dataset
_original_init = AudioKwsDataset.__init__

def _cached_init(self, root_dir, **kwargs):
    self.__dict__.update(_cached_dataset.__dict__)

AudioKwsDataset.__init__ = _cached_init

Loading 51093 audio files into RAM...
  5000/51093 loaded...
  10000/51093 loaded...
  15000/51093 loaded...
  20000/51093 loaded...
  25000/51093 loaded...
  30000/51093 loaded...
  35000/51093 loaded...
  40000/51093 loaded...
  45000/51093 loaded...
  50000/51093 loaded...
Done! All 51093 spectrograms cached in RAM.


## DEFINING THE LENET MODEL

In [ ]:
class Lenet5(nn.Module):
    def __init__(self, num_classes):
        super(Lenet5, self).__init__()
        # Input: (1, 40, 101)
        self.conv1 = nn.Conv2d(1, 6, kernel_size=5, padding=2)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv2 = nn.Conv2d(6, 16, kernel_size=5)

        # CALCULATION FOR FLATTEN SIZE:
        # After Conv1 + Pool: (20, 50)
        # After Conv2: (16, 46) -> Pool: (8, 23)
        # 8 * 23 * 16 channels = 2944
        self.fc1 = nn.Linear(16 * 8 * 23, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, num_classes)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = x.view(x.size(0), -1) # Flatten logic
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)
        return x

In [ ]:
full_dataset = AudioKwsDataset(root_dir='/content/v0.01_train')

In [ ]:
# Get the dictionary from the original dataset and flip it to index:name
# Your AudioKwsDataset has label_to_idx = {'yes': 0, 'no': 1, ...}
idx_to_label = {v: k for k, v in full_dataset.label_to_idx.items()}

# Create a sorted list based on the indices (0, 1, 2... 18)
class_names = [idx_to_label[i] for i in range(len(idx_to_label))]
print(f"Found {len(class_names)} classes: {class_names[:5]}...")

Found 31 classes: ['_silence_', 'bed', 'bird', 'cat', 'dog']...


##INSTALLING TORCHDEC

## UPLOADING ENTIRE DATASET TO COLAB LOCAL

In [ ]:
import shutil
import os

# 1. Define paths
drive_path = '/content/v0.01_train' # Update this if needed!
local_path = drive_path

# 2. Check if the source path on Drive exists
if not os.path.exists(drive_path):
    print(f"Error: Source directory on Google Drive not found: {drive_path}")
    print("Please ensure Google Drive is mounted and the path to your dataset is correct.")
else:
    # 3. Copy the entire folder from Drive to Local Colab SSD
    if not os.path.exists(local_path):
        print("Moving data to local SSD for high-speed training...")
        try:
            shutil.copytree(drive_path, local_path)
            print("Transfer complete!")
        except Exception as e:
            print(f"Error during data transfer: {e}")
            print("Please check permissions or disk space.")
    else:
        print("Data already exists on local SSD.")

# Final check to confirm local path existence
if os.path.exists(local_path):
    print(f"Local data directory '{local_path}' is accessible.")
    # Optionally, list some contents to verify
    # print(f"Contents: {os.listdir(local_path)[:5]}...")
else:
    print(f"Local data directory '{local_path}' is NOT accessible after copy attempt.")


Data already exists on local SSD.
Local data directory '/content/v0.01_train' is accessible.


## TRAINING THE MODEL

In [ ]:
from tqdm.notebook import tqdm

# 1. Configuration - CRITICAL FOR SPEED
# Using num_workers > 0 tells Colab to use multiple CPU cores for audio math
# Using pin_memory=True speeds up the transfer from CPU RAM to GPU RAM
num_cpus = os.cpu_count()
print(f"Using {num_cpus} CPU cores for preprocessing...")

device = 'cuda' if torch.cuda.is_available() else 'cpu'
data_path = '/content/v0.01_train' # Corrected path to the downloaded dataset

# 2. Initialize and Split
full_dataset = AudioKwsDataset(root_dir=data_path)
num_classes = len(full_dataset.label_to_idx)
train_size = int(0.8 * len(full_dataset)) # generator
test_size = len(full_dataset) - train_size
train_data, test_data = random_split(full_dataset, [train_size, test_size], generator=torch.Generator().manual_seed(42))

# SPEED FIX: Added num_workers and pin_memory
train_loader = DataLoader(
    train_data,
    batch_size=128,          # Increased batch size to 128 for faster GPU throughput
    shuffle=True,
    num_workers=num_cpus,    # Uses all available CPU cores for FFT math
    pin_memory=True          # Speeds up CPU-to-GPU data transfer
)

test_loader = DataLoader(
    test_data,
    batch_size=128,
    shuffle=False,
    num_workers=num_cpus,
    pin_memory=True
)

# 3. Initialize Model
model = Lenet5(num_classes=num_classes).to(device)
optimizer = Adam(model.parameters(), lr=0.001)
loss_fn = nn.CrossEntropyLoss()

# 4. Optimized Training Loop
if not os.path.exists('./res'): os.mkdir('./res')

for epoch in range(30): # Start with 10 to verify speed; then increase
    model.train()
    total_loss = 0

    # Progress bar that updates every batch
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}", leave=False)

    for specs, labels in pbar:
        specs, labels = specs.to(device, non_blocking=True), labels.to(device, non_blocking=True)

        optimizer.zero_grad()
        outputs = model(specs)
        loss = loss_fn(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    # Evaluation
    model.eval()
    correct = 0
    with torch.no_grad():
        for spec, lbl in test_loader:
            spec, lbl = spec.to(device), lbl.to(device)
            pred = model(spec).argmax(dim=1)
            correct += (pred == lbl).sum().item()

    acc = 100 * correct / len(test_data)
    print(f"Epoch {epoch+1} | Loss: {total_loss/len(train_loader):.4f} | Acc: {acc:.2f}%")
    torch.save(model.state_dict(), '/content/res/kws_model_best.pt')

Using 2 CPU cores for preprocessing...


Epoch 1:   0%|          | 0/320 [00:00<?, ?it/s]

Epoch 1 | Loss: 1.7955 | Acc: 74.49%


Epoch 2:   0%|          | 0/320 [00:00<?, ?it/s]

Epoch 2 | Loss: 0.6689 | Acc: 81.62%


...

Epoch 28 | Loss: 0.0339 | Acc: 87.73%


Epoch 29:   0%|          | 0/320 [00:00<?, ?it/s]

Epoch 29 | Loss: 0.0370 | Acc: 88.30%


Epoch 30:   0%|          | 0/320 [00:00<?, ?it/s]

Epoch 30 | Loss: 0.0283 | Acc: 88.42%


##PREDICTING AUDIO AND TESTING ACCURACY

In [ ]:
import torch
from torch.utils.data import DataLoader
import torch.nn as nn
import os

# Import the classes we defined in previous blocks
# (In Colab, make sure these classes are in the same notebook or imported)
# from models import Lenet5
# from dataprocess import AudioKwsDataset

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'

def test_model(model_path, data_path):
    # 1. Initialize the Dataset (Same way as training)
    full_dataset = AudioKwsDataset(root_dir=data_path)
    num_classes = len(full_dataset.label_to_idx)

    # 2. Re-create the 20% split so we test on the same 'unseen' data
    train_size = int(0.8 * len(full_dataset))
    test_size = len(full_dataset) - train_size
    _, test_dataset = torch.utils.data.random_split(
        full_dataset, [train_size, test_size],
        generator=torch.Generator().manual_seed(42) # Seed ensures the same split
    )

    test_dataloader = DataLoader(test_dataset, batch_size=128, shuffle=False)

    # 3. Load the model architecture and the saved weights
    model = Lenet5(num_classes=num_classes)
    model.load_state_dict(torch.load(model_path))
    model.to(device)
    model.eval()

    print(f"Testing model: {model_path} on {len(test_dataset)} samples...")

    pred_tags = []
    true_tags = []

    with torch.no_grad():
        for batch in test_dataloader:
            batch_data = batch[0].to(device)
            batch_label = batch[1].to(device)

            logits = model(batch_data)
            pred = torch.argmax(logits, dim=1).cpu().numpy()
            tags = batch_label.cpu().numpy()

            pred_tags.extend(pred)
            true_tags.extend(tags)

    # 4. Calculate Accuracy
    correct_num = sum(int(x == y) for (x, y) in zip(pred_tags, true_tags))
    accuracy = correct_num / len(pred_tags)

    print(f"Final Test Accuracy: {accuracy * 100:.2f}%")
    return accuracy

if __name__ == '__main__':
    # Adjust paths to match your Colab setup
    BEST_MODEL_PATH = '/content/res/kws_model_best.pt'
    DATA_DIR = local_path

    if os.path.exists(BEST_MODEL_PATH):
        test_model(BEST_MODEL_PATH, DATA_DIR)
    else:
        print("Error: Model file not found. Did training finish?")

Testing model: /content/res/kws_model_best.pt on 10219 samples...
Final Test Accuracy: 88.42%


In [ ]:
# 3. Initialize Model
model = Lenet5(num_classes=num_classes).to(device)
optimizer = Adam(model.parameters(), lr=0.001)
loss_fn = nn.CrossEntropyLoss()

In [ ]:
import torch
import torchaudio
import torchaudio.transforms as T
import torch.nn.functional as F
import os

def verify_model_prediction(sample_idx, model_path, dataset_obj):
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    # 1. Get the Ground Truth (Expected Result)
    audio_path, true_label_id = dataset_obj.samples[sample_idx]
    idx_to_label = {v: k for k, v in dataset_obj.label_to_idx.items()}
    expected_word = idx_to_label[true_label_id]

    # 2. Load the Model and Prediction Logic
    num_classes = len(dataset_obj.label_to_idx)
    model = Lenet5(num_classes=num_classes).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()

    # 3. Preprocess Audio
    waveform, sr = torchaudio.load(audio_path)
    if sr != 16000:
        waveform = torchaudio.functional.resample(waveform, sr, 16000)
    if waveform.shape[1] > 16000:
        waveform = waveform[:, :16000]
    else:
        waveform = torch.nn.functional.pad(waveform, (0, 16000 - waveform.shape[1]))

    mel_transform = T.MelSpectrogram(
        sample_rate=16000, n_fft=400, win_length=400, hop_length=160, n_mels=40
    ).to(device)

    spec = mel_transform(waveform.to(device))
    spec = torch.log(spec + 1e-9).unsqueeze(0)

    # 4. Predict
    with torch.no_grad():
        logits = model(spec)
        probs = F.softmax(logits, dim=1)
        conf, pred_idx = torch.max(probs, dim=1)
        predicted_word = idx_to_label[pred_idx.item()]

    # 5. Show Results
    print("="*30)
    print(f"SAMPLE ID: {sample_idx}")
    print(f"FILE: {os.path.basename(audio_path)}")
    print("-" * 30)
    print(f"EXPECTED (True):  {expected_word}")
    print(f"PREDICTED:       {predicted_word}")
    print(f"CONFIDENCE:      {conf.item()*100:.2f}%")
    print("-" * 30)

    if expected_word == predicted_word:
        print("✅ SUCCESS: Prediction matches!")
    else:
        print("❌ FAILURE: Prediction is incorrect.")
    print("="*30)

# --- EXECUTION ---
# You can change 'sample_idx' to any number to test different files
verify_model_prediction(sample_idx=0, model_path='/content/res/kws_model_best.pt', dataset_obj=full_dataset)

SAMPLE ID: 0
FILE: pink_noise.wav
------------------------------
EXPECTED (True):  _silence_
PREDICTED:       _silence_
CONFIDENCE:      95.45%
------------------------------
✅ SUCCESS: Prediction matches!


##SAVING THE DISTILLATION PARAMETERS IN SOFTWARE

In [ ]:
import torch
import numpy as np
import os

# 1. Configuration - Update these to match your Colab files
# Using the path where you saved the best model earlier
MODEL_PATH = '/content/res/kws_model_best.pt'
OUTPUT_FILE = '/content/res/lenet5_parameters_kws_float.txt'

# 2. Get the number of classes (folders) dynamically
# This ensures Lenet5(num_classes=X) matches your weights
data_path = '/content/v0.01_train'
all_dirs = sorted([d for d in os.listdir(data_path) if os.path.isdir(os.path.join(data_path, d))])
if "_unknown_" in all_dirs: all_dirs.remove("_unknown_")
num_classes = len(all_dirs)

# 2. Set the class count to match your CHECKPOINT (the error says 19)
num_classes = 31

# 3. Initialize Model with 31 classes
model = Lenet5(num_classes=31)
model.load_state_dict(torch.load(MODEL_PATH, map_location='cpu'))
model.eval()
# 19
# Set numpy to print full arrays without truncation
np.set_printoptions(threshold=np.inf, linewidth=100)

print(f"Exporting floating-point parameters for {num_classes} classes...")

# 4. Save Parameters
with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
    for name, params in model.named_parameters():
        print(f"Layer: {name} | Shape: {params.shape}")

        # Write layer metadata
        f.write(name + ':' + str(params.shape) + '\n')

        # Write the raw weights as floats
        # We use detach().numpy() to move them from PyTorch to NumPy
        f.write(str(params.detach().numpy()) + '\n')
        f.write("\n" + "="*50 + "\n")

print(f"Successfully saved to {OUTPUT_FILE}")

Exporting floating-point parameters for 31 classes...
Layer: conv1.weight | Shape: torch.Size([6, 1, 5, 5])
Layer: conv1.bias | Shape: torch.Size([6])
Layer: conv2.weight | Shape: torch.Size([16, 6, 5, 5])
Layer: conv2.bias | Shape: torch.Size([16])
Layer: fc1.weight | Shape: torch.Size([120, 2944])
Layer: fc1.bias | Shape: torch.Size([120])
Layer: fc2.weight | Shape: torch.Size([84, 120])
Layer: fc2.bias | Shape: torch.Size([84])
Layer: fc3.weight | Shape: torch.Size([31, 84])
Layer: fc3.bias | Shape: torch.Size([31])
Successfully saved to /content/res/lenet5_parameters_kws_float.txt


## DISTILLATION CODE

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
from tqdm.notebook import tqdm
import os

# 1. SETUP MODELS
def run_distillation(EPOCHS=50, lr=0.001, batch_size=64, alpha=0.3, T=7):
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    # Initialize Dataset (Use your local SSD path for speed)
    full_dataset = AudioKwsDataset(root_dir=local_path)
    num_classes = len(full_dataset.label_to_idx)

    train_size = int(0.8 * len(full_dataset))
    test_size = len(full_dataset) - train_size
    train_data, test_data = random_split(full_dataset, [train_size, test_size], generator=torch.Generator().manual_seed(42))

    train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
    test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False, num_workers=4)

    # TEACHER: The High-Performance Model
    from torchvision.models import resnet18
    teacher_model = resnet18(pretrained=True)
    # Modify ResNet for 1-channel Mel-spectrogram input
    teacher_model.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
    teacher_model.fc = nn.Linear(512, num_classes)
    teacher_model.to(device).eval()

    # STUDENT: Your LeNet-5 for FPGA
    student_model = Lenet5(num_classes=num_classes).to(device)

    # 2. LOSS FUNCTIONS & OPTIMIZER
    optimizer = torch.optim.Adam(student_model.parameters(), lr=lr)

    # Standard Loss for the correct label
    hard_loss_fn = nn.CrossEntropyLoss()

    # Distillation Loss to match teacher's soft probabilities
    soft_loss_fn = nn.KLDivLoss(reduction='batchmean')

    # 3. TRAINING LOOP
    max_test_acc = 0.
    for epoch in range(EPOCHS):
        student_model.train()
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}", leave=False)

        for specs, labels in pbar:
            specs, labels = specs.to(device), labels.to(device)

            # Get Teacher predictions (No gradients needed for teacher)
            with torch.no_grad():
                teacher_logits = teacher_model(specs)

            # Student predictions
            student_logits = student_model(specs)

            # --- DISTILLATION MATH ---
            # Hard loss: compare student results to the ground truth
            loss_hard = hard_loss_fn(student_logits, labels)

            # Soft loss: compare student's smoothed output to teacher's smoothed output
            # We multiply by (T*T) to keep the gradients at the same scale
            loss_soft = soft_loss_fn(
                F.log_softmax(student_logits / T, dim=1),
                F.softmax(teacher_logits / T, dim=1)
            ) * (T * T)

            # Final combined loss based on alpha
            loss = alpha * loss_hard + (1 - alpha) * loss_soft

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            pbar.set_postfix(loss=f"{loss.item():.4f}")

        # 4. EVALUATION
        student_model.eval()
        correct = 0
        with torch.no_grad():
            for specs, labels in test_loader:
                specs, labels = specs.to(device), labels.to(device)
                preds = student_model(specs).argmax(dim=1)
                correct += (preds == labels).sum().item()

        acc = 100 * correct / len(test_data)
        if acc > max_test_acc:
            max_test_acc = acc
            torch.save(student_model.state_dict(), '/content/res/distilled_lenet5_best.pt')
            print(f"Best Distilled Model Saved! Acc: {acc:.2f}%")

    print(f"Distillation Finished. Peak Accuracy: {max_test_acc:.2f}%")

if __name__ == '__main__':
    run_distillation(EPOCHS=30, alpha=0.3, T=5)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 205MB/s]


Epoch 1:   0%|          | 0/639 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


...

Epoch 27:   0%|          | 0/639 [00:00<?, ?it/s]

Epoch 28:   0%|          | 0/639 [00:00<?, ?it/s]

Epoch 29:   0%|          | 0/639 [00:00<?, ?it/s]

Epoch 30:   0%|          | 0/639 [00:00<?, ?it/s]

Distillation Finished. Peak Accuracy: 88.38%


##QUANTIFICATION

In [ ]:
import torch
from torch.utils.data import DataLoader, random_split

# 1. Define the local path (ensure you use the local SSD path for speed)
data_path = local_path

# 2. Instantiate the full dataset using your defined class
# This uses the specific settings: 40 Mels, 400 Win, 160 Hop
full_dataset = AudioKwsDataset(root_dir=data_path)

# 3. Create the same Train/Test split used during training
# Seed is set to 42 to ensure the split is identical every time you run it
train_size = int(0.8 * len(full_dataset))
test_size = len(full_dataset) - train_size
train_data, test_data = random_split(full_dataset, [train_size, test_size], generator=torch.Generator().manual_seed(42))

# 4. Initialize DataLoaders
# num_workers=4 speeds up audio processing; pin_memory=True speeds up GPU transfer
batch_size = 64
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)

# 5. Verification Output
print(f"Dataset Successfully Loaded!")
print(f"Total Samples: {len(full_dataset)}")
print(f"Training Samples: {len(train_data)}")
print(f"Testing Samples: {len(test_data)}")
print(f"Number of Classes: {len(full_dataset.label_to_idx)}")
print("-" * 30)
print(f"Class Mapping: {full_dataset.label_to_idx}")

Dataset Successfully Loaded!
Total Samples: 51093
Training Samples: 40874
Testing Samples: 10219
Number of Classes: 31
------------------------------
Class Mapping: {'_silence_': 0, 'bed': 1, 'bird': 2, 'cat': 3, 'dog': 4, 'down': 5, 'eight': 6, 'five': 7, 'four': 8, 'go': 9, 'happy': 10, 'house': 11, 'left': 12, 'marvin': 13, 'nine': 14, 'no': 15, 'off': 16, 'on': 17, 'one': 18, 'right': 19, 'seven': 20, 'sheila': 21, 'six': 22, 'stop': 23, 'three': 24, 'tree': 25, 'two': 26, 'up': 27, 'wow': 28, 'yes': 29, 'zero': 30}


In [ ]:
import struct
import os
import torch
import numpy as np

def float_to_hex_le(f):
    """Converts a float to a 16-bit Little-Endian Hex string using standard libraries."""
    # '<e' means Little-Endian (<) and Half-Precision float (e)
    return struct.pack('>e', f).hex()

# Ensure model is in eval mode and on CPU
model.eval()
PARAM_DIR = "/content/res/parameters/"

with torch.no_grad():
    for name, module in model.named_modules():
        if isinstance(module, (torch.nn.Conv2d, torch.nn.Linear)):
            file_path = os.path.join(PARAM_DIR, f"{name}.txt")
            print(f"Re-quantizing {name} (Correct Method)...")

            with open(file_path, "w") as f:
                # 1. Weights - NO TRANSPOSE to verify software accuracy first
                weights = module.weight.detach().cpu().numpy().flatten()
                for val in weights:
                    f.write(float_to_hex_le(val))

                # 2. Biases - Appended at the end of the same file
                if module.bias is not None:
                    biases = module.bias.detach().cpu().numpy().flatten()
                    for val in biases:
                        f.write(float_to_hex_le(val))

print("Files regenerated. Now run the verification script.")

Re-quantizing conv1 (Correct Method)...
Re-quantizing conv2 (Correct Method)...
Re-quantizing fc1 (Correct Method)...
Re-quantizing fc2 (Correct Method)...
Re-quantizing fc3 (Correct Method)...
Files regenerated. Now run the verification script.


In [ ]:
import torch
import torch.nn as nn
import struct
import os
import numpy as np

# 1. SETUP PATHS AND MODEL
PARAM_DIR = "/content/res/parameters/"
MODEL_PATH = '/content/res/distilled_lenet5_best.pt'

if not os.path.exists(PARAM_DIR):
    os.makedirs(PARAM_DIR)

# Initialize your specific LeNet-5 architecture
# Note: num_classes must match your dataset (19 based on previous chat)
quant_model = Lenet5(num_classes=31)
quant_model.load_state_dict(torch.load(MODEL_PATH, map_location='cpu'))
quant_model.eval()

def float_to_hex_fp16(f):
    """
    Converts a float to 16-bit Big-Endian Hex.
    '>e' ensures the sign bit is at the far left (bit 15),
    matching your Verilog: sign = floatA[15]
    """
    return struct.pack('>e', f).hex()

# 2. QUANTIZATION & EXPORT LOOP
print("Starting Quantization to 16-bit Hex...")

with torch.no_grad():
    for name, module in quant_model.named_modules():
        # Target only layers with trainable parameters
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            file_path = os.path.join(PARAM_DIR, f"{name}.txt")

            # Extract weights and biases
            w_flat = module.weight.detach().cpu().numpy().flatten()
            b_flat = module.bias.detach().cpu().numpy().flatten() if module.bias is not None else []

            print(f"Exporting {name}: {len(w_flat)} weights, {len(b_flat)} biases...")

            with open(file_path, "w", encoding="utf-8") as f:
                # Write weights first
                for val in w_flat:
                    f.write(float_to_hex_fp16(val) + "\n") # \n makes it $readmemh compatible

                # Append biases immediately after
                for val in b_flat:
                    f.write(float_to_hex_fp16(val) + "\n")

print(f"Export Complete. Files saved in: {PARAM_DIR}")

Starting Quantization to 16-bit Hex...
Exporting conv1: 150 weights, 6 biases...
Exporting conv2: 2400 weights, 16 biases...
Exporting fc1: 353280 weights, 120 biases...
Exporting fc2: 10080 weights, 84 biases...
Exporting fc3: 2604 weights, 31 biases...
Export Complete. Files saved in: /content/res/parameters/


In [ ]:
from torch.utils.data import DataLoader

# Create a loader with num_workers=0 for stable single-sample inference
simple_test_loader = DataLoader(
    test_data,
    batch_size=1,        # Set to 1 for easier single-audio testing
    shuffle=True,        # Shuffle to get a different sample each time
    num_workers=0,       # 0 ensures no multi-processing hangs
    pin_memory=False
)

print(f"simple_test_loader defined with {len(simple_test_loader)} samples.")

simple_test_loader defined with 10219 samples.


In [ ]:
import torch
import torch.nn as nn
import struct
import os
import numpy as np

# 1. SETUP DEVICE AND MODEL
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Initialize your LeNet-5
quant_model = Lenet5(num_classes=31)

# Load your distilled weights (adjust path as needed)
MODEL_PATH = '/content/res/distilled_lenet5_best.pt'
if os.path.exists(MODEL_PATH):
    quant_model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
    print(" Distilled weights loaded.")

# CRITICAL: Move model to the correct device
quant_model.to(device)
quant_model.eval()

# 2. DEFINE QUANTIZATION LOGIC
PARAM_DIR = "/content/res/parameters/"
if not os.path.exists(PARAM_DIR): os.makedirs(PARAM_DIR)

def float_to_hex_fp16(f):
    """
    Converts float to 16-bit Big-Endian Hex.
    Matches Verilog: sign = bit[15], exponent = bits[14:10], mantissa = bits[9:0]
    """
    return struct.pack('>e', f).hex()

# 3. EXPORT LOOP
print(" Exporting weights and biases to Verilog-ready Hex...")

with torch.no_grad():
    for name, module in quant_model.named_modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            file_path = os.path.join(PARAM_DIR, f"{name}.txt")

            # Move to CPU for numpy conversion
            w_flat = module.weight.detach().cpu().numpy().flatten()
            b_flat = module.bias.detach().cpu().numpy().flatten() if module.bias is not None else []

            with open(file_path, "w", encoding="utf-8") as f:
                # Write Weights
                for val in w_flat:
                    f.write(float_to_hex_fp16(val) + "\n")
                # Append Biases
                for val in b_flat:
                    f.write(float_to_hex_fp16(val) + "\n")

            print(f"  - {name}: {len(w_flat)} weights + {len(b_flat)} biases saved.")

# 4. FINAL ACCURACY VERIFICATION (Handling the FloatTensor/CudaTensor error)
def final_accuracy_check(model, loader):
    model.eval()
    correct = 0
    total = 0

    print("\n Verifying Quantized Accuracy...")
    with torch.no_grad():
        for specs, labels in loader:
            # FIX: Move inputs to the SAME device as the model
            specs = specs.to(device)
            labels = labels.to(device)

            outputs = model(specs)
            preds = outputs.argmax(dim=1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)
            if total >= 500: break # Quick verification

    print(f"Final Software Accuracy: {100 * correct / total:.2f}%")

# Use your existing loader
final_accuracy_check(quant_model, test_loader)

Using device: cuda
 Distilled weights loaded.
 Exporting weights and biases to Verilog-ready Hex...
  - conv1: 150 weights + 6 biases saved.
  - conv2: 2400 weights + 16 biases saved.
  - fc1: 353280 weights + 120 biases saved.
  - fc2: 10080 weights + 84 biases saved.
  - fc3: 2604 weights + 31 biases saved.

 Verifying Quantized Accuracy...
Final Software Accuracy: 87.30%


## CHECKING ERROR BETWEEN TRUE AND QUANTISED VALUE

In [ ]:
import torch
import struct
import numpy as np

def analyze_quantization(model, layer_name='conv1', num_samples=5):
    model.eval()
    # Get the layer module
    module = dict(model.named_modules())[layer_name]

    # Get true weights (Float32) # 19
    true_weights = module.weight.detach().cpu().numpy().flatten()

    print(f"Quantization Analysis for Layer: {layer_name}")
    print(f"{'True Value (F32)':>18} | {'Hex (FP16)':>10} | {'Quantized Value':>18} | {'Error %':>10}")
    print("-" * 65)

    for i in range(num_samples):
        true_val = true_weights[i]

        # 1. Convert to 16-bit Big-Endian Hex (Quantization)
        hex_val = struct.pack('>e', true_val).hex()

        # 2. Convert back to float to see the "Quantized Value"
        # This is what the FPGA actually "sees"
        quant_val = np.frombuffer(bytes.fromhex(hex_val), dtype='>f2').astype(np.float32)[0]

        # 3. Calculate Precision Loss (Error)
        error = abs(true_val - quant_val)
        error_pct = (error / abs(true_val) * 100) if true_val != 0 else 0

        print(f"{true_val:18.8f} | {hex_val:>10} | {quant_val:18.8f} | {error_pct:8.4f}%")

# Run the comparison
analyze_quantization(quant_model, layer_name='conv2')

Quantization Analysis for Layer: conv2
  True Value (F32) | Hex (FP16) |    Quantized Value |    Error %
-----------------------------------------------------------------
        0.12111259 |       2fc0 |         0.12109375 |   0.0156%
        0.01887323 |       24d5 |         0.01887512 |   0.0100%
        0.01414338 |       233e |         0.01414490 |   0.0107%
       -0.13819814 |       b06c |        -0.13818359 |   0.0105%
       -0.22008124 |       b30b |        -0.22009277 |   0.0052%


## EXPECTED VS PREDICTED LABEL (QUANTISED)

In [ ]:
import torch
import random
import struct
import os

def final_quantized_check(model, dataset_obj, loader):
    model.eval()
    model.to(device) # Ensure it's on the right device (GPU/CPU)

    # 1. Get the mapping from Index to Label Name
    idx_to_label = {v: k for k, v in dataset_obj.label_to_idx.items()}

    # 2. Grab a random sample from the test set
    # Using the loader ensures the Log-Mel transform is applied correctly
    specs, labels = next(iter(loader))
    idx = random.randint(0, len(specs) - 1)

    input_tensor = specs[idx].unsqueeze(0).to(device)
    true_idx = labels[idx].item()

    # 3. Predict
    with torch.no_grad():
        output = model(input_tensor)
        pred_idx = torch.argmax(output, dim=1).item()

    # 4. Show Results
    print("="*45)
    print(f"{'CATEGORY':<15} | {'VALUE':<20}")
    print("-" * 45)
    print(f"{'Expected Word':<15} | {idx_to_label[true_idx]} (ID: {true_idx})")
    print(f"{'Predicted Word':<15} | {idx_to_label[pred_idx]} (ID: {pred_idx})")
    print("="*45)

    if true_idx == pred_idx:
        print("SUCCESS: The quantized model is 100% accurate for this sample.")
    else:
        print("MISMATCH: Check your bias loading or hex order.")

    # 5. GENERATE VERILOG INPUT STIMULUS
    print("\n--- Verilog Testbench Hex Input (First 5 Pixels) ---")
    # Pull pixels back to CPU for formatting
    pixels = input_tensor.flatten().cpu().numpy()
    for i in range(5):
        # '>e' = 16-bit Big-Endian Hex (matches your floatAdd16 input)
        hex_val = struct.pack('>e', pixels[i]).hex()
        print(f"tb_input[{i}] = 16'h{hex_val}; // Float: {pixels[i]:.6f}")

# Execute the final check
# We use full_dataset for labels and simple_test_loader for the audio sample
final_quantized_check(quant_model, full_dataset, simple_test_loader)

CATEGORY        | VALUE               
---------------------------------------------
Expected Word   | five (ID: 7)
Predicted Word  | five (ID: 7)
SUCCESS: The quantized model is 100% accurate for this sample.

--- Verilog Testbench Hex Input (First 5 Pixels) ---
tb_input[0] = 16'hc6b9; // Float: -6.723228
tb_input[1] = 16'hc313; // Float: -3.537685
tb_input[2] = 16'hc51b; // Float: -5.105752
tb_input[3] = 16'hc514; // Float: -5.079855
tb_input[4] = 16'hc3bf; // Float: -3.872789
